# 05 · Retrieval-Augmented Generation (RAG)

**Objective.** Ground responses in company policy by retrieving relevant chunks and injecting them into the prompt - the single biggest lever against hallucination.

**Advanced technique covered:** RAG (dense embeddings + ChromaDB). **Basic technique covered:** TF-IDF knowledge-base matching (sparse baseline for comparison).

>  Logic lives in `src/`; these notebooks orchestrate, experiment, visualise and analyse (good-practice separation, ULO6). Run the notebooks in order `01 -> 07` after completing setup in `README.md`.

In [1]:
# --- bootstrap: make `import config` and `from src import ...` work from anywhere ---
import sys, os
from pathlib import Path
p = Path.cwd()
ROOT = next((c for c in [p, *p.parents] if (c/'config.py').exists() and (c/'src').exists()), p)
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
print('project root:', ROOT)

project root: C:\Users\lenovo\Desktop\ANLP_8420_GROUPC\Assignment3_GroupC\Codes


In [2]:
import warnings
warnings.filterwarnings("ignore")

## 1. Knowledge Base

A small company knowledge base was created to support retrieval-augmented generation (RAG). The knowledge base consists of customer service policy documents covering topics such as orders, refunds, payments, cancellations, subscriptions, and account management.

Before indexing, the documents are divided into smaller chunks. Chunking allows the retrieval system to return only the most relevant pieces of information rather than entire documents, improving retrieval precision and reducing unnecessary context passed to the LLM.

In [3]:
from src.rag import load_knowledge_documents, RAGStore, format_context
records = load_knowledge_documents()
print('chunks:', len(records))
print('sample chunk:\n', records[0]['text'][:200])

chunks: 214
sample chunk:
 # Account & Subscription Policy

## Account management
Customers can update their email, password, delivery address, and contact details
from Account Settings. Address changes apply to future orders o


## 2. Dense Retrieval (Sentence Transformers + ChromaDB)

Dense retrieval is implemented using sentence embeddings stored in a ChromaDB vector database.

Each knowledge chunk is converted into a semantic embedding and indexed for efficient similarity search. When a customer submits a question, the query is embedded using the same model and matched against the stored knowledge base.

Unlike keyword-based retrieval, dense retrieval can identify semantically related content even when the wording differs between the customer query and the company documentation.

In [4]:
store = RAGStore()
if store.count() == 0:
    store.build(records)
for c in store.retrieve('how long does a refund take after I return an item?'):
    print(f"({c['score']}) {c['source']}: {c['text'][:80]}...")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


(0.7711) refund_policy.md: # Refund Policy

## Eligibility
Refunds are available within 30 days of delivery...
(0.7673) faq_refund.md: In most cases, refunds are issued within a few business days after the return of...
(0.7392) faq_refund.md: Generally, refunds are processed within a few business days after the request is...
(0.6535) refund_policy.md: ## Shipping-fee refunds
Original shipping fees are refunded only when the return...


## 3. Sparse Retrieval Baseline (TF-IDF)

To evaluate the benefits of semantic retrieval, a traditional TF-IDF retriever is implemented as a baseline.

TF-IDF relies on exact term overlap between the query and the knowledge base documents. Comparing TF-IDF against dense retrieval helps determine whether semantic embeddings improve retrieval quality for customer support questions that may use varied wording.

In [5]:
from src.tfidf_retrieval import TfidfRetriever
sparse = TfidfRetriever(); sparse.build(records)
q = 'can I cancel an order before it ships?'
print('DENSE :', [c['source'] for c in store.retrieve(q)])
print('SPARSE:', [c['source'] for c in sparse.retrieve(q)])

[tfidf-retriever] indexed 214 chunks
DENSE : ['faq_order.md', 'cancellation_policy.md', 'faq_order.md', 'faq_refund.md']
SPARSE: ['faq_order.md', 'payment_policy.md', 'cancellation_policy.md', 'faq_subscription.md']


## Analysis

The retrieval results demonstrate a key advantage of dense retrieval over TF-IDF. While TF-IDF performs well when the customer query contains the same keywords as the knowledge base, dense retrieval is better able to handle paraphrased questions and semantic similarity.

For example, a customer asking about receiving "money back" may still retrieve refund-related documents even if the exact word "refund" is not used. This makes dense retrieval particularly useful in customer service environments where users often describe the same issue using different language.

The comparison also highlights the trade-off between retrieval quality and computational cost, as dense retrieval requires embedding generation and vector indexing.

## 4. End-to-End Grounded Answer (RAG-on vs RAG-off)

To evaluate the impact of retrieval augmentation, responses are generated both with and without access to the company knowledge base.

In the RAG-on setting, relevant policy documents are retrieved and included in the prompt before response generation. In the RAG-off setting, the model receives only the customer message without supporting company knowledge.

Comparing these outputs demonstrates how retrieval grounding influences response quality, factual accuracy, and policy compliance.

In [6]:
from src.generation import generate_response
msg = 'My order 48213 is two weeks late, what are my options?'
ctx_on = format_context(store.retrieve(msg))
on  = generate_response(msg, ctx_on, variant='few_shot')
off = generate_response(msg, '(no company knowledge provided)', variant='few_shot')
print('RAG-ON :', on['reply'])
print('\nRAG-OFF:', off['reply'])

RAG-ON : I'm sorry your order #48213 is running late. To provide an accurate estimate of delivery time, could you please provide us with your Order Number or Tracking Number? With this information, we will be able to check the shipping details and provide you with the expected delivery date.

RAG-OFF: I'm sorry your order #48213 is running late. Standard delivery is 3-5 business days; since yours is past that window I've flagged it for our delivery team and you're eligible to request a shipping refund if it passes 7 business days.


## Limitations

The quality of RAG responses depends heavily on the quality and coverage of the knowledge base. If relevant information is missing or poorly represented, retrieval performance will decrease and the generated response may be incomplete.

In addition, retrieval errors can propagate into the generation stage, causing the model to rely on irrelevant context. Future improvements could include reranking models, hybrid retrieval approaches, and larger domain-specific knowledge collections.

## Summary

This notebook implemented a complete retrieval-augmented generation (RAG) pipeline for the customer service system.

The experiments included:
- Knowledge base construction and chunking
- Dense retrieval using sentence embeddings and ChromaDB
- Sparse TF-IDF retrieval as a baseline
- Comparison of retrieval approaches
- Grounded response generation using retrieved company knowledge

The results demonstrate that retrieval augmentation improves the factual grounding of generated responses and helps align customer service answers with company policies.

**Next:** Notebook 06 integrates retrieval, classification, sentiment analysis, NER, and escalation logic into a unified agentic workflow and user-facing system.